In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG ecommerce_project")
spark.sql("USE SCHEMA sales")

In [0]:
%run ./00_transformations

In [0]:
raw_orders = spark.table("raw_orders")
enriched_orders = build_enriched_orders(
    raw_orders, spark.table("enriched_customers"), spark.table("enriched_products")
)
save_table(enriched_orders, "enriched_orders", expected_min_rows=9994)

In [0]:
e = spark.table("enriched_orders")
r = spark.table("raw_orders")

print("rows enriched / raw :", e.count(), r.count())                       # both 9994
print("null dates          :", e.filter("order_date IS NULL").count())     # 0
print("years               :", sorted([x[0] for x in e.select("year").distinct().collect()]))  # 2014-2017
print("null category rows  :", e.filter("category IS NULL").count())       # 204
print("profit on null-category rows:", e.filter("category IS NULL").agg(F.round(F.sum("profit"), 2)).first()[0])
print("profit enriched/raw :", e.agg(F.round(F.sum("profit"), 2)).first()[0],
                               r.agg(F.round(F.sum("profit"), 2)).first()[0])
display(e.limit(5))

## Note 9: Orders with no matching product (204 rows)

- **Notebook:** 03_enriched_orders
- **Finding:** 204 orders (44 distinct product IDs) have no row in Products. They carry 1,999.23 of the total 278,417.03 profit.
- **Decision:** Use a left join so no orders and no profit are lost. Category and sub-category are null for these rows. They appear as a NULL group in the aggregates.
- **Check:** Total profit of enriched orders must equal total profit of raw orders.
- **Result:** null-category rows = 204; profit on those rows = 1,999.23; enriched rows = 9,994 vs raw rows = 9,994; profit enriched = 278,417.03 vs raw = 278,417.03. All match.



## Note 10: Date format is d/M/yyyy

- **Notebook:** 03_enriched_orders
- **Finding:** `order_date` and `ship_date` are strings such as `21/8/2016` and `6/10/2016`, with no leading zeros.
- **Fix:** Parse with `to_date(order_date, "d/M/yyyy")` and derive `year`. Using `dd/MM/yyyy` would return null for these values.
- **Check:** No null dates after parsing. Years should be 2014 to 2017.
- **Result:** null dates = 0; years found = 2014, 2015, 2016, 2017.